In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# Load the dataset
df = pd.read_excel(r'cleaned_dataset_with_iso_forest.xlsx', sheet_name='Ori Cleaned Data')

# Define features and target
X = df[['AGE', 'PC1', 'PC1_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 
        'AEA', 'WR_HR', 'WR', 'ACC', 'VOID', 'SLUMP',  'APPLICATION_ID', 'TOTAL_BINDER', 'TOTAL_AGG', 
        'w/b', 'b/a', 'SCM%', 'FAGG%', 'CAGG%']].values

# Assuming 'Chloride Ion Penetrability' is now continuous
y = df['COULOMB_TEST'].values

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)  # Normalize input features
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()  # Normalize target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=500, random_state=42, max_depth=30, min_samples_leaf=2, min_samples_split=2)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)

# XGBoost Regressor
xgb_reg = XGBRegressor(random_state=42, colsample_bytree=1, learning_rate=0.2, max_depth=3, n_estimators=100, subsample=1)
xgb_reg.fit(X_train, y_train)
y_pred_xgb = xgb_reg.predict(X_test)
r2_xgb = r2_score(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)

# Artificial Neural Network (MLP Regressor)
ann_reg = MLPRegressor(hidden_layer_sizes=(150,), max_iter=500, random_state=42, activation='relu', learning_rate='constant')
ann_reg.fit(X_train, y_train)
y_pred_ann = ann_reg.predict(X_test)
r2_ann = r2_score(y_test, y_pred_ann)
mse_ann = mean_squared_error(y_test, y_pred_ann)

# Displaying the results
print("Random Forest Regressor:")
print(f"R²: {r2_rf:.2f}")
print(f"Mean Squared Error: {mse_rf:.2f}\n")

print("XGBoost Regressor:")
print(f"R²: {r2_xgb:.2f}")
print(f"Mean Squared Error: {mse_xgb:.2f}\n")

print("ANN Regressor:")
print(f"R²: {r2_ann:.2f}")
print(f"Mean Squared Error: {mse_ann:.2f}\n")


Random Forest Regressor:
R²: 0.15
Mean Squared Error: 0.81

XGBoost Regressor:
R²: 0.14
Mean Squared Error: 0.81

ANN Regressor:
R²: 0.04
Mean Squared Error: 0.92



In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Load the dataset
df = pd.read_excel(r'chloride_Classification.xlsx', sheet_name='Sheet1')

# Define features and target
X = df[['AGE', 'PC1', 'PC2', 'PC1_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 
        'AEA', 'WR_HR', 'WR', 'ACC', 'VOID', 'FIBER', 'SLUMP', 'IS_PUMP', 'IS_LATEX', 
        'LATEX_AMOUNT', 'IS_LIGHT_WEIGHT', 'APPLICATION_ID', 'TOTAL_BINDER', 'TOTAL_AGG', 
        'w/b', 'b/a', 'SCM%', 'FAGG%', 'CAGG%', 'FA%', 'SS%', 'SF%']].values

# Assuming 'Chloride Ion Penetrability' is now continuous
y = df['COULOMB_TEST'].values

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Random Forest Regressor Grid Search
rf_param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_param_grid, cv=3, n_jobs=-1, scoring='r2')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)

# 2. XGBoost Regressor Grid Search
xgb_param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

xgb_grid = GridSearchCV(XGBRegressor(random_state=42), xgb_param_grid, cv=3, n_jobs=-1, scoring='r2')
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
r2_xgb = r2_score(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)

# 3. MLP Regressor Grid Search
mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (150,)],
    'activation': ['relu', 'tanh'],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [300, 500]
}

mlp_grid = GridSearchCV(MLPRegressor(random_state=42), mlp_param_grid, cv=3, n_jobs=-1, scoring='r2')
mlp_grid.fit(X_train, y_train)
best_mlp = mlp_grid.best_estimator_
y_pred_mlp = best_mlp.predict(X_test)
r2_mlp = r2_score(y_test, y_pred_mlp)
mse_mlp = mean_squared_error(y_test, y_pred_mlp)

# Displaying the results
print("Best Random Forest Regressor:")
print(f"R²: {r2_rf:.2f}")
print(f"Mean Squared Error: {mse_rf:.2f}")
print(f"Best Parameters: {rf_grid.best_params_}\n")

print("Best XGBoost Regressor:")
print(f"R²: {r2_xgb:.2f}")
print(f"Mean Squared Error: {mse_xgb:.2f}")
print(f"Best Parameters: {xgb_grid.best_params_}\n")

print("Best ANN (MLP) Regressor:")
print(f"R²: {r2_mlp:.2f}")
print(f"Mean Squared Error: {mse_mlp:.2f}")
print(f"Best Parameters: {mlp_grid.best_params_}\n")


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/zhangtianjie/opt/ana

Best Random Forest Regressor:
R²: 0.15
Mean Squared Error: 1857228.16
Best Parameters: {'max_depth': 30, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}

Best XGBoost Regressor:
R²: 0.14
Mean Squared Error: 1875914.54
Best Parameters: {'colsample_bytree': 1, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1}

Best ANN (MLP) Regressor:
R²: 0.13
Mean Squared Error: 1900895.12
Best Parameters: {'activation': 'relu', 'hidden_layer_sizes': (150,), 'learning_rate': 'constant', 'max_iter': 500}



/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
